# Goal Rescue Re-tuning (Phase 7)

Re-derives `coeff_multiplier` and `s_min` values after the real-world stat anchoring
changed the baseline scoring distributions. The mechanism (Lever C quality-scaled
bonus) was proven correct in `goal_rescue_investigation.ipynb` — this notebook
only sweeps the constants against the new baseline.

**NS_QUALITY_STATS** are computed here in the correct pre-sigmoid space
(std of `processed_raw_score - scoring_dot_contribution`) with mean fixed at 0.0.

All sweep tables show four groups:
- `bad_not_scored` / `good_not_scored` — reference baselines (never change; not affected by fix)
- `bad_scored` / `good_scored` — after applying the fix
- `bad_gap` / `good_gap` — (scored − not_scored): the goal bonus contribution, which is what we're tuning

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np

project_root = Path("..").resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.services.analytics.match_ratings_service import MatchRatingsService

TEAM_NAME = "Valencia CF"
MATCHES_PATH = project_root / "tests" / "fixtures" / "testing_data" / "valencia_cf_1" / "matches.json"

POSITION_GROUP_MAP = {
    'ST': 'ST', 'LW': 'Winger', 'RW': 'Winger', 'CM': 'CM',
    'CDM': 'CDM', 'CB': 'CB', 'LB': 'Fullback', 'RB': 'Fullback',
}
POSITIONS_ANALYSED = ['ST', 'Winger', 'CM']

BASE_COEFF        = {'ST': 1.5, 'Winger': 1.3, 'CM': 1.0}
BASE_ASSIST_COEFF = {'ST': 1.1, 'Winger': 1.0, 'CM': 0.75}
K = 1.5

with open(project_root / "config" / "performance_weights.json")    as f: weights    = json.load(f)
with open(project_root / "config" / "performance_means_stds.json") as f: means_stds = json.load(f)
with open(MATCHES_PATH) as f: data = json.load(f)
print(f"Loaded {len(data)} matches")

Loaded 155 matches


In [2]:
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def _scoring_dot_contribution(z_scores, weights):
    return (z_scores.get('goals_p90_z', 0.0) * weights[0]
            + z_scores.get('assists_p90_z', 0.0) * weights[1])

class GoalRescueCaptureService(MatchRatingsService):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._cur = {}
        self.last_capture = None

    def reset_capture(self):
        self._cur = {}
        self.last_capture = None

    def _calculate_dot_product(self, z_scores, weights):
        raw = super()._calculate_dot_product(z_scores, weights)
        self._cur['dot_product'] = raw
        self._cur['scoring_dot_contribution'] = _scoring_dot_contribution(z_scores, weights)
        return raw

    def _effective_goal_bonus(self, goals, shots, coeff):
        res = super()._effective_goal_bonus(goals, shots, coeff)
        self._cur['goal_bonus_pre_iso'] = res
        return res

    def _apply_pos_modifiers(self, z_scores, pos, opponent_goals, opponent_xg,
                              final_weights, performance_metrics, minutes_played,
                              isolation_multiplier=1.0):
        processed, event_bonus = super()._apply_pos_modifiers(
            z_scores=z_scores, pos=pos, opponent_goals=opponent_goals,
            opponent_xg=opponent_xg, final_weights=final_weights,
            performance_metrics=performance_metrics, minutes_played=minutes_played,
            isolation_multiplier=isolation_multiplier)
        self._cur.update({
            'processed_raw_score': processed, 'event_bonus': event_bonus,
            'isolation_multiplier': isolation_multiplier,
            'goals': performance_metrics.get('goals', 0),
            'shots': performance_metrics.get('shots', 0),
            'minutes_played': minutes_played,
        })
        self.last_capture = dict(self._cur)
        return processed, event_bonus

print("GoalRescueCaptureService defined.")

GoalRescueCaptureService defined.


In [3]:
cap_service = GoalRescueCaptureService(weights, means_stds)
records = []

for match in data:
    mo = match['data']; hl = mo['half_length']
    is_home = mo.get('home_team_name') == TEAM_NAME
    team_xg = (mo.get('home_stats', {}) if is_home else mo.get('away_stats', {})).get('xg', 0)
    opp_xg  = (mo.get('away_stats', {}) if is_home else mo.get('home_stats', {})).get('xg', 0)
    supremacy = cap_service._calculate_match_supremacy_scalar(team_xg=team_xg, xg_against=opp_xg)

    for perf in match['player_performances']:
        if perf['performance_type'] != 'Outfield': continue
        positions = perf.get('positions_played', [])
        if len(positions) != 1: continue
        group = POSITION_GROUP_MAP.get(positions[0])
        if group not in POSITIONS_ANALYSED: continue

        cap_service.reset_capture()
        rating = cap_service.calculate_outfield_rating(perf, mo, hl, TEAM_NAME)
        cap = cap_service.last_capture
        if rating is None or cap is None: continue

        minutes = cap['minutes_played']
        impact  = float(np.sqrt(min(minutes, 90.0) / 90.0))
        goal_contribution   = cap['goal_bonus_pre_iso'] * cap['isolation_multiplier']
        assist_contribution = cap['event_bonus'] - goal_contribution

        records.append({
            'match_id': match['id'], 'player_id': perf['player_id'],
            'group': group, 'rating': rating,
            'goals': cap['goals'], 'shots': cap['shots'], 'minutes': minutes,
            'processed': cap['processed_raw_score'], 'event_bonus': cap['event_bonus'],
            'goal_contribution': goal_contribution, 'assist_contribution': assist_contribution,
            'impact': impact, 'isolation': cap['isolation_multiplier'],
            'supremacy': supremacy, 'goal_coeff': cap.get('goal_coeff', 0),
            'scoring_dot_contribution': cap['scoring_dot_contribution'],
        })

df = pd.DataFrame(records)
df['scored'] = df['goals'] >= 1
df['corrected_non_scoring'] = df['processed'] - df['scoring_dot_contribution']
df['corrected_pctile']   = df.groupby('group')['corrected_non_scoring'].rank(pct=True) * 100
df['corrected_bad_half'] = df['corrected_pctile'] < 50

print(f"Captured {len(df)} single-position performances")
print(df.groupby('group').agg(n=('rating','size'), scored=('scored','sum'),
                              mean_rating=('rating','mean')).round(3))

Captured 1102 single-position performances
          n  scored  mean_rating
group                           
CM      424      90        6.670
ST      234      95        7.157
Winger  444     106        6.762


In [4]:
NS_QUALITY_STATS = {}
for g, grp in df.groupby('group'):
    computed_mean = grp['corrected_non_scoring'].mean()
    std           = grp['corrected_non_scoring'].std(ddof=1)
    NS_QUALITY_STATS[g] = (0.0, std)
    print(f"{g:8s}  computed mean={computed_mean:+.4f}  std={std:.4f}")

CM        computed mean=+0.1031  std=0.4071
ST        computed mean=+0.2136  std=0.4448
Winger    computed mean=-0.0147  std=0.3595


In [5]:
def reconstruct(row):
    raw = row['processed'] * row['impact'] + row['event_bonus']
    r   = cap_service._apply_sigmoid_transformation(raw_score=raw) - row['supremacy']
    return round(max(0.0, min(10.0, r)), 1)

def goal_pre_iso(goals, shots, coeff):
    return coeff * max(goals - shots * cap_service.XG_PER_SHOT, goals * cap_service.GOAL_FLOOR_RATE)

df['reconstructed'] = df.apply(reconstruct, axis=1)
diff = (df['rating'] - df['reconstructed']).abs()
print(f"Reconstruction check: max diff={diff.max():.6f}  n>1e-6={(diff>1e-6).sum()} / {len(df)}")
assert diff.max() < 1e-6, "Reconstruction mismatch."

Reconstruction check: max diff=0.000000  n>1e-6=0 / 1102


In [6]:
# ── Baseline: all four groups, no fix applied ─────────────────────────────────
# not_scored values are the reference — they never change under any configuration.
# gap = scored − not_scored within the same quality tier = pure goal bonus contribution.

print("Production baseline (no fix):")
print(f"{'Group':<8} {'bad_not_scored':>15} {'bad_scored':>11} {'bad_gap':>8} "
      f"{'good_not_scored':>16} {'good_scored':>12} {'good_gap':>9}")
print("-" * 85)
for g in POSITIONS_ANALYSED:
    bns = df[(df['group']==g) & df['corrected_bad_half']  & ~df['scored']]['rating'].mean()
    bs  = df[(df['group']==g) & df['corrected_bad_half']  &  df['scored']]['rating'].mean()
    gns = df[(df['group']==g) & ~df['corrected_bad_half'] & ~df['scored']]['rating'].mean()
    gs  = df[(df['group']==g) & ~df['corrected_bad_half'] &  df['scored']]['rating'].mean()
    print(f"{g:<8} {bns:>15.2f} {bs:>11.2f} {bs-bns:>8.2f} "
          f"{gns:>16.2f} {gs:>12.2f} {gs-gns:>9.2f}")

Production baseline (no fix):
Group     bad_not_scored  bad_scored  bad_gap  good_not_scored  good_scored  good_gap
-------------------------------------------------------------------------------------
ST                  5.88        8.05     2.17             6.63         8.85      2.22
Winger              5.86        7.51     1.65             6.82         8.65      1.82
CM                  5.90        7.61     1.71             6.71         8.43      1.72


In [7]:
# ── Predict function ──────────────────────────────────────────────────────────

def predict(row, s_min_by_group: dict, coeff_mult_by_group: dict, k: float = K) -> float:
    mean, std = NS_QUALITY_STATS[row['group']]
    z     = (row['corrected_non_scoring'] - mean) / std
    s_min = s_min_by_group[row['group']]
    qs    = s_min + (1 - s_min) * _sigmoid(k * z)
    coeff = coeff_mult_by_group[row['group']] * BASE_COEFF[row['group']]
    new_goal  = goal_pre_iso(row['goals'], row['shots'], coeff) * row['isolation'] * qs
    new_event = new_goal + row['assist_contribution']
    raw       = row['processed'] * row['impact'] + new_event
    r         = cap_service._apply_sigmoid_transformation(raw_score=raw) - row['supremacy']
    return round(max(0.0, min(10.0, r)), 1)

def group_means(g, s_map, c_map, k=K):
    """Return (bad_ns, bad_s, good_ns, good_s) for one position group.
    not_scored means are always from original ratings (they can't change)."""
    bad_s   = df[(df['group']==g) & df['corrected_bad_half']  &  df['scored']]
    good_s  = df[(df['group']==g) & ~df['corrected_bad_half'] &  df['scored']]
    bad_ns  = df[(df['group']==g) & df['corrected_bad_half']  & ~df['scored']]['rating'].mean()
    good_ns = df[(df['group']==g) & ~df['corrected_bad_half'] & ~df['scored']]['rating'].mean()
    bad_s_r  = bad_s.apply(lambda r: predict(r, s_map, c_map, k), axis=1).mean()
    good_s_r = good_s.apply(lambda r: predict(r, s_map, c_map, k), axis=1).mean()
    return bad_ns, bad_s_r, good_ns, good_s_r

print("Predict function defined.")

Predict function defined.


In [8]:
# ── CM sweep (coeff only, s_min=0.55) ────────────────────────────────────────
S_MIN_CM = 0.55
CM_COEFF_CANDIDATES = [0.67, 0.55, 0.45, 0.35, 0.30, 0.25]

# Reference not_scored (constant regardless of fix)
cm_bad_ns  = df[(df['group']=='CM') & df['corrected_bad_half']  & ~df['scored']]['rating'].mean()
cm_good_ns = df[(df['group']=='CM') & ~df['corrected_bad_half'] & ~df['scored']]['rating'].mean()
print(f"CM reference (not affected by fix):")
print(f"  bad_not_scored={cm_bad_ns:.2f}   good_not_scored={cm_good_ns:.2f}")
print(f"  (gap to beat: bad_scored should exceed bad_not_scored by a reasonable margin)")
print()

rows = []
for cm_mult in CM_COEFF_CANDIDATES:
    s_map = {'ST': 0.55, 'Winger': 0.55, 'CM': S_MIN_CM}
    c_map = {'ST': 0.67, 'Winger': 0.67, 'CM': cm_mult}
    bns, bs, gns, gs = group_means('CM', s_map, c_map)
    rows.append({
        'cm_coeff_mult': cm_mult,
        'eff_coeff': round(cm_mult * BASE_COEFF['CM'], 3),
        'bad_scored': round(bs, 2),
        'bad_gap': round(bs - bns, 2),
        'good_scored': round(gs, 2),
        'good_gap': round(gs - gns, 2),
    })

print(pd.DataFrame(rows).set_index('cm_coeff_mult').to_string())

CM reference (not affected by fix):
  bad_not_scored=5.90   good_not_scored=6.71
  (gap to beat: bad_scored should exceed bad_not_scored by a reasonable margin)

               eff_coeff  bad_scored  bad_gap  good_scored  good_gap
cm_coeff_mult                                                       
0.67                0.67        6.92     1.02         8.07      1.36
0.55                0.55        6.79     0.89         7.96      1.25
0.45                0.45        6.68     0.78         7.85      1.14
0.35                0.35        6.58     0.68         7.75      1.04
0.30                0.30        6.52     0.62         7.69      0.98
0.25                0.25        6.43     0.53         7.63      0.92


In [9]:
# ── ST and Winger 2D sweep (coeff × s_min) ───────────────────────────────────
# not_scored reference printed once per position; sweep shows scored + gap.

CM_COEFF_FINAL = 0.30  # <-- update once you've picked it above

ST_COEFF_GRID  = [0.25, 0.30, 0.35, 0.40, 0.45]
ST_SMIN_GRID   = [0.55, 0.65, 0.75, 0.85, 0.95, 1.00]
WNG_COEFF_GRID = [0.35, 0.40, 0.45, 0.50, 0.55, 0.67]
WNG_SMIN_GRID  = [0.55, 0.65, 0.75, 0.85, 0.95, 1.00]

for pos, coeff_grid, smin_grid in [
    ('ST',     ST_COEFF_GRID,  ST_SMIN_GRID),
    ('Winger', WNG_COEFF_GRID, WNG_SMIN_GRID),
]:
    pos_bad_ns  = df[(df['group']==pos) & df['corrected_bad_half']  & ~df['scored']]['rating'].mean()
    pos_good_ns = df[(df['group']==pos) & ~df['corrected_bad_half'] & ~df['scored']]['rating'].mean()

    print(f"\n{'='*70}")
    print(f"{pos} reference (not affected by fix):")
    print(f"  bad_not_scored={pos_bad_ns:.2f}   good_not_scored={pos_good_ns:.2f}")
    print()

    bad_s_rows, bad_gap_rows, good_s_rows, good_gap_rows = [], [], [], []

    for coeff in coeff_grid:
        bad_s_row  = {'coeff': coeff}
        bad_g_row  = {'coeff': coeff}
        good_s_row = {'coeff': coeff}
        good_g_row = {'coeff': coeff}

        for s_min in smin_grid:
            s_map = {'ST': 0.55, 'Winger': 0.55, 'CM': S_MIN_CM}
            c_map = {'ST': 0.67, 'Winger': 0.67, 'CM': CM_COEFF_FINAL}
            s_map[pos] = s_min
            c_map[pos] = coeff
            _, bs, _, gs = group_means(pos, s_map, c_map)
            bad_s_row[s_min]  = round(bs, 2)
            bad_g_row[s_min]  = round(bs - pos_bad_ns, 2)
            good_s_row[s_min] = round(gs, 2)
            good_g_row[s_min] = round(gs - pos_good_ns, 2)

        bad_s_rows.append(bad_s_row);   bad_gap_rows.append(bad_g_row)
        good_s_rows.append(good_s_row); good_gap_rows.append(good_g_row)

    bad_s_piv   = pd.DataFrame(bad_s_rows).set_index('coeff')
    bad_gap_piv = pd.DataFrame(bad_gap_rows).set_index('coeff')
    good_s_piv  = pd.DataFrame(good_s_rows).set_index('coeff')
    good_gap_piv= pd.DataFrame(good_gap_rows).set_index('coeff')

    print("bad_scored:")
    print(bad_s_piv.to_string())
    print("\nbad_gap (= bad_scored − bad_not_scored, goal bonus contribution for poor performers):")
    print(bad_gap_piv.to_string())
    print("\ngood_scored:")
    print(good_s_piv.to_string())
    print("\ngood_gap (= good_scored − good_not_scored, goal bonus contribution for good performers):")
    print(good_gap_piv.to_string())


ST reference (not affected by fix):
  bad_not_scored=5.88   good_not_scored=6.63

bad_scored:
       0.55  0.65  0.75  0.85  0.95   1.0
coeff                                    
0.25   6.38  6.41  6.45  6.48  6.50  6.52
0.30   6.47  6.52  6.55  6.59  6.63  6.65
0.35   6.56  6.61  6.66  6.70  6.75  6.78
0.40   6.66  6.71  6.77  6.82  6.86  6.89
0.45   6.76  6.80  6.87  6.91  6.98  7.01

bad_gap (= bad_scored − bad_not_scored, goal bonus contribution for poor performers):
       0.55  0.65  0.75  0.85  0.95   1.0
coeff                                    
0.25   0.50  0.53  0.56  0.60  0.62  0.64
0.30   0.59  0.64  0.67  0.70  0.75  0.77
0.35   0.68  0.73  0.78  0.82  0.86  0.90
0.40   0.78  0.83  0.89  0.94  0.98  1.01
0.45   0.88  0.92  0.99  1.03  1.10  1.13

good_scored:
       0.55  0.65  0.75  0.85  0.95   1.0
coeff                                    
0.25   7.96  7.96  7.97  7.98  7.98  7.98
0.30   8.03  8.04  8.05  8.06  8.06  8.07
0.35   8.11  8.11  8.12  8.13  8.14  8.14
0.40  

In [10]:
# ── k sweep — controls steepness of quality discrimination ───────────────────
# Hold coeff at your preferred values. Lower k → flatter q → less spread.
# not_scored values are still constant; gaps show the goal bonus contribution.

K_SWEEP_COEFF = {'ST': 0.25, 'Winger': 0.35, 'CM': 0.25}  # update from grid above
K_SWEEP_SMIN  = {'ST': 0.55, 'Winger': 0.55, 'CM': 0.55}
K_CANDIDATES  = [0.25, 0.50, 0.75, 1.00, 1.25, 1.50]

# Reference not_scored per position (constant)
ns_ref = {}
for g in POSITIONS_ANALYSED:
    ns_ref[g] = (
        df[(df['group']==g) & df['corrected_bad_half']  & ~df['scored']]['rating'].mean(),
        df[(df['group']==g) & ~df['corrected_bad_half'] & ~df['scored']]['rating'].mean(),
    )

print("Reference (not_scored, constant):")
for g in POSITIONS_ANALYSED:
    print(f"  {g:<8} bad_not_scored={ns_ref[g][0]:.2f}  good_not_scored={ns_ref[g][1]:.2f}")
print()

rows = []
for k_val in K_CANDIDATES:
    row = {'k': k_val}
    for g in POSITIONS_ANALYSED:
        _, bs, _, gs = group_means(g, K_SWEEP_SMIN, K_SWEEP_COEFF, k=k_val)
        bns, gns = ns_ref[g]
        row[f'{g}_bad_s']   = round(bs, 2)
        row[f'{g}_bad_gap'] = round(bs - bns, 2)
        row[f'{g}_good_s']  = round(gs, 2)
        row[f'{g}_good_gap']= round(gs - gns, 2)
    rows.append(row)

print(f"Coeff: ST={K_SWEEP_COEFF['ST']}  Winger={K_SWEEP_COEFF['Winger']}  CM={K_SWEEP_COEFF['CM']}")
print(f"s_min: {K_SWEEP_SMIN['ST']} (shared)")
print()
print(pd.DataFrame(rows).set_index('k').to_string())

Reference (not_scored, constant):
  ST       bad_not_scored=5.88  good_not_scored=6.63
  Winger   bad_not_scored=5.86  good_not_scored=6.82
  CM       bad_not_scored=5.90  good_not_scored=6.71

Coeff: ST=0.25  Winger=0.35  CM=0.25
s_min: 0.55 (shared)

      ST_bad_s  ST_bad_gap  ST_good_s  ST_good_gap  Winger_bad_s  Winger_bad_gap  Winger_good_s  Winger_good_gap  CM_bad_s  CM_bad_gap  CM_good_s  CM_good_gap
k                                                                                                                                                             
0.25      6.38         0.5       7.91         1.28          6.34            0.49           7.73             0.91      6.46        0.56       7.59         0.89
0.50      6.38         0.5       7.92         1.29          6.32            0.47           7.75             0.93      6.45        0.55       7.60         0.90
0.75      6.38         0.5       7.94         1.30          6.32            0.46           7.76             0.9

In [11]:
# ── Final configuration ───────────────────────────────────────────────────────
FINAL_COEFF_MULT = {'ST': 0.35, 'Winger': 0.35, 'CM': 0.35}  # <-- update
FINAL_S_MIN      = {'ST': 0.55, 'Winger': 0.55, 'CM': 0.55}  # <-- update
FINAL_K          = 0.25                                            # <-- update if changed

print("FINAL CONFIGURATION")
print(f"  k = {FINAL_K}")
for g in POSITIONS_ANALYSED:
    eff = FINAL_COEFF_MULT[g] * BASE_COEFF[g]
    print(f"  {g:<8} coeff_mult={FINAL_COEFF_MULT[g]}  s_min={FINAL_S_MIN[g]}  "
          f"effective_goal_coeff={eff:.3f} (was {BASE_COEFF[g]})")
print()

print(f"{'Group':<8} {'bad_ns':>7} {'bad_s':>7} {'bad_gap':>8} "
      f"{'good_ns':>8} {'good_s':>7} {'good_gap':>9}")
print("-" * 60)
for g in POSITIONS_ANALYSED:
    bns, bs, gns, gs = group_means(g, FINAL_S_MIN, FINAL_COEFF_MULT, k=FINAL_K)
    ns_b, ns_g = ns_ref[g]
    print(f"{g:<8} {ns_b:>7.2f} {bs:>7.2f} {bs-ns_b:>8.2f} "
          f"{ns_g:>8.2f} {gs:>7.2f} {gs-ns_g:>9.2f}")

# Non-scorers untouched check
ns = df[~df['scored']]
ns_after = ns.apply(lambda r: predict(r, FINAL_S_MIN, FINAL_COEFF_MULT, FINAL_K), axis=1)
print(f"\nNon-scorer check: max |before-after| = {(ns['rating'] - ns_after).abs().max():.6f} (must be 0.0)")

FINAL CONFIGURATION
  k = 0.25
  ST       coeff_mult=0.35  s_min=0.55  effective_goal_coeff=0.525 (was 1.5)
  Winger   coeff_mult=0.35  s_min=0.55  effective_goal_coeff=0.455 (was 1.3)
  CM       coeff_mult=0.35  s_min=0.55  effective_goal_coeff=0.350 (was 1.0)

Group     bad_ns   bad_s  bad_gap  good_ns  good_s  good_gap
------------------------------------------------------------
ST          5.88    6.57     0.69     6.63    8.04      1.41
Winger      5.86    6.34     0.49     6.82    7.73      0.91
CM          5.90    6.59     0.69     6.71    7.70      0.99

Non-scorer check: max |before-after| = 0.000000 (must be 0.0)


In [12]:
# ── Phase 8 constants output ──────────────────────────────────────────────────
print("Constants ready for Phase 8 service patch:")
print()
print("NS_QUALITY_STATS = {")
for g, (m, s) in NS_QUALITY_STATS.items():
    print(f'    "{g}": ({m}, {s:.6f}),')
print("}")
print(f"\nK = {FINAL_K}")
print()
print("COEFF_MULTIPLIER = {")
for g, v in FINAL_COEFF_MULT.items():
    print(f'    "{g}": {v},')
print("}")
print()
print("S_MIN = {")
for g, v in FINAL_S_MIN.items():
    print(f'    "{g}": {v},')
print("}")

Constants ready for Phase 8 service patch:

NS_QUALITY_STATS = {
    "CM": (0.0, 0.407052),
    "ST": (0.0, 0.444797),
    "Winger": (0.0, 0.359505),
}

K = 0.25

COEFF_MULTIPLIER = {
    "ST": 0.35,
    "Winger": 0.35,
    "CM": 0.35,
}

S_MIN = {
    "ST": 0.55,
    "Winger": 0.55,
    "CM": 0.55,
}
